In [2]:
#get all the tables:
from db_setup import conn, currencyexchange, customer, date, product, sales, store
#instead of saving csv files in sql and then importing one by one
import pandas as pd

customer_summary=pd.read_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\customer_summary.parquet")


c:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\db_setup.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  currencyexchange = pd.read_sql("SELECT * FROM currencyexchange", conn)
c:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\db_setup.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer = pd.read_sql("SELECT * FROM customer", conn)
c:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\db_setup.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  date = pd.read_sql("SELECT * FR

In [3]:
#the base columns:
rfm = customer_summary[["customerkey", "recency_days", "order_count", "lifetime_revenue"]].copy()

In [ ]:
#get the percent_rank to allow scoring:
rfm["recency_pct"] = rfm["recency_days"].rank(method="min", pct=True, ascending=True)
rfm["frequency_pct"] = rfm["order_count"].rank(method="min", pct=True, ascending=False)
rfm["monetary_pct"] = rfm["lifetime_revenue"].rank(method="min", pct=True, ascending=False)



In [ ]:
#lets get the scores. 

def score_from_pct(p):
    if p < 0.20:
        return 5
    elif p < 0.40:
        return 4
    elif p < 0.60:
        return 3
    elif p < 0.80:
        return 2
    else:
        return 1
    

rfm["recency_score"] = rfm["recency_pct"].apply(score_from_pct)



def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x == 3:
        return 3
    elif x == 4:
        return 4
    else:
        return 5

rfm["frequency_score"] = rfm["order_count"].apply(frequency_score)


rfm["monetary_score"] = rfm["monetary_pct"].apply(score_from_pct)


In [7]:
#RFM scores and combined rfm score

rfm["rfm_score"] = (
    rfm["recency_score"].astype(str) +
    rfm["frequency_score"].astype(str) +
    rfm["monetary_score"].astype(str)
)

rfm["rfm_total_score"] = (
    rfm["recency_score"] +
    rfm["frequency_score"] +
    rfm["monetary_score"]
)

In [8]:
#rfm.to_csv("rfm.csv",index=False)  - saved this result

rfm

,customerkey,recency_days,order_count,lifetime_revenue,recency_pct,frequency_pct,monetary_pct,recency_score,frequency_score,monetary_score,rfm_score,rfm_total_score
0,15,1139,1,1299.708307,0.671772,0.443490,0.662618,2,1,2,212,5
1,180,236,2,1103.346104,0.179845,0.162325,0.698668,5,2,2,522,9
2,185,1785,1,666.453820,0.791602,0.443490,0.786590,2,1,2,212,5
3,243,2893,1,148.903542,0.967022,0.443490,0.927092,1,1,1,111,3
4,387,156,3,2341.268364,0.121183,0.053206,0.513064,5,3,3,533,11
...,...,...,...,...,...,...,...,...,...,...,...,...
49482,2099619,1380,4,6709.935970,0.690889,0.015358,0.197446,2,4,5,245,11
49483,2099656,74,4,10404.677800,0.052074,0.015358,0.095075,5,4,5,545,14
49484,2099697,585,1,38.201100,0.433366,0.443490,0.973468,3,1,1,311,5
49485,2099711,2441,2,6008.670000,0.929315,0.162325,0.228100,1,2,4,124,7


In [9]:
rfm.to_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\rfm.parquet", index=False)

In [26]:
#Boundaries:

boundaries = {}

# recency — ascending
for p, label in [(0.20, "p20"), (0.40, "p40"), (0.60, "p60"), (0.80, "p80")]:
    boundaries[f"recency_{label}"] = rfm["recency_days"].quantile(p)

# frequency — descending means we flip the percentile
for p, label in [(0.20, "p20"), (0.40, "p40"), (0.60, "p60"), (0.80, "p80")]:
    boundaries[f"frequency_{label}"] = rfm["order_count"].quantile(1 - p)

# monetary — descending
for p, label in [(0.20, "p20"), (0.40, "p40"), (0.60, "p60"), (0.80, "p80")]:
    boundaries[f"monetary_{label}"] = rfm["lifetime_revenue"].quantile(1 - p)

pd.DataFrame([boundaries])


,recency_p20,recency_p40,recency_p60,recency_p80,frequency_p20,frequency_p40,frequency_p60,frequency_p80,monetary_p20,monetary_p40,monetary_p60,monetary_p80
0,268.2,542.0,858.0,1810.0,2.0,2.0,1.0,1.0,6652.996375,3415.654996,1701.258304,606.961784
